# Lezione 08 — Contenitori e CI/CD

**Corso:** MLOps e Monitoraggio con Python · **Blocco B**
**Tempo stimato:** 110 minuti · **Capitolo del manuale:** `manual/08_contenedores_cicd_IT.md`

## Cosa vedrai in questa lezione

- Scrivere un **`Dockerfile`** reale per pacchettizzare l'API della lezione 7, e capire quale problema di riproducibilità risolve che fissare le versioni con `requirements.txt` (lezione 2) NON risolve.
- Scrivere un flusso reale di **GitHub Actions** (`ci.yml`) che installa le dipendenze, esegue la suite di test e, se passano, "registra" il modello.
- Validare per davvero, in Python, la sintassi e la struttura di entrambi i file, senza aver bisogno di Docker né di GitHub installati.
- Eseguire localmente, senza Docker, l'unica parte genuinamente eseguibile: i test che la CI lancerebbe.
- Usare il **registro dei modelli di MLflow** (alias e tag) per marcare una versione come candidata, in produzione o archiviata.
- Capire la differenza tra **CI** e **CD**, e perché in ML conviene un CD più cauto.

> **Attenzione (limite di questo ambiente):** Docker e GitHub Actions non si possono eseguire per davvero dentro Colab né dentro questo ambiente di validazione. Questo notebook genera e valida il contenuto reale di entrambi i file, ed esegue localmente solo la parte genuinamente eseguibile senza di essi (i test). La sezione 9 del capitolo spiega il comando esatto per provare il flusso completo fuori da qui.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

## Prerequisiti

- Dalla lezione 7: l'API FastAPI (`Cliente`, `/predecir`, `/predecir_lote`) che questo capitolo pacchettizza.
- Dalla lezione 6: l'idea di una suite di test con `pytest` (unitari, sui dati, sul modello).
- Dalla lezione 4: `mlflow.sklearn.log_model` e l'idea di un modello registrato come artefatto versionato.
- Dalla lezione 2: `requirements.txt` con versioni esatte, e perché non basta a fissare l'ambiente completo.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (incluso FastAPI, httpx, pytest, PyYAML e MLflow).

In [ ]:
import os

# Fissiamo queste variabili PRIMA della prima volta che si importa mlflow (anche il controllo qui sotto),
# perché nessun messaggio automatico di MLflow dipenda dall'ora di sistema nell'output di questo notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("uvicorn", "uvicorn")
asegurar("pytest", "pytest")
asegurar("PyYAML", "yaml")
asegurar("mlflow", "mlflow")
print("Librerie pronte.")

---
## Cella 0 — Setup

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import json
import logging
import warnings

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from importlib.metadata import version
import yaml

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

# Come nella lezione 4: fissiamo queste variabili PRIMA di importare mlflow e alziamo il suo logger a ERROR
# (WARNING non basta: mlflow.utils.requirements_utils avvisa in WARNING con dettagli non deterministici)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS


def ejecutar_pytest(argumentos):
    """Esegue pytest con subprocess.run e restituisce un riepilogo SENZA i tempi di esecuzione (cambiano a ogni esecuzione)."""
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q"] + argumentos, capture_output=True, text=True)
    salida = resultado.stdout + resultado.stderr
    m_passed = re.search(r"(\d+) passed", salida)
    m_failed = re.search(r"(\d+) failed", salida)
    m_error = re.search(r"(\d+) error", salida)
    return {
        "returncode": resultado.returncode,
        "passed": int(m_passed.group(1)) if m_passed else 0,
        "failed": int(m_failed.group(1)) if m_failed else 0,
        "error": int(m_error.group(1)) if m_error else 0,
    }


print("Librerie e funzioni ausiliarie pronte. Versione di MLflow:", mlflow.__version__)

---
## Cartella di lavoro effimera

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(os.path.join(CARPETA, "tests"))
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Riaddestrare e salvare l'artefatto (come nella lezione 7)

L'API che pacchettizzeremo ha bisogno di un `modelo.pkl` reale su disco.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# La stessa configurazione vincitrice delle lezioni 4, 5 e 7: regressione logistica, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
F1 = round(f1_score(y_test, y_pred), 4)
AUC = round(roc_auc_score(y_test, y_proba), 4)
print("F1 (set di test):", F1)
print("AUC (set di test):", AUC)
assert F1 > 0.25 and AUC > 0.6, "il ri-addestramento non ha riprodotto il modello atteso"

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
print("Artefatto salvato in:", RUTA_MODELO)

---
## 2. `requirements.txt`: necessario, ma non sufficiente

Come nella lezione 2, generiamo `requirements.txt` con le versioni **realmente installate**, con `importlib.metadata.version`. La differenza è l'ambito: qui vanno solo le librerie di cui l'API ha bisogno in TEMPO DI ESECUZIONE dentro il contenitore.

In [ ]:
# Come nella lezione 2 (importlib.metadata.version, mai scritte a mano), ma ora le 6 librerie
# di cui l'API della lezione 7 ha bisogno in TEMPO DI ESECUZIONE dentro il contenitore: non include pytest né mlflow,
# che sono strumenti di sviluppo/CI, non dipendenze del servizio che si distribuisce
PAQUETES_API = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "scikit-learn"),
    ("joblib", "joblib"), ("fastapi", "fastapi"), ("uvicorn", "uvicorn"),
]
versiones_api = {nombre_pip: version(nombre_pip) for nombre_pip, _ in PAQUETES_API}

RUTA_REQUIREMENTS = os.path.join(CARPETA, "requirements.txt")
with open(RUTA_REQUIREMENTS, "w") as f:
    f.write("\n".join(f"{paquete}=={ver}" for paquete, ver in versiones_api.items()) + "\n")

print("requirements.txt generato:")
print(open(RUTA_REQUIREMENTS).read())

> **Attenzione:** `requirements.txt` fissa QUALI pacchetti Python e a quale versione, ma non fissa: la versione di **Python** stessa, il **sistema operativo** (Debian? Alpine? Windows?), le **librerie C di sistema** da cui dipendono sotto il cofano pacchetti come `numpy` o `scikit-learn` (BLAS/LAPACK), né l'**architettura del processore** (x86_64 contro ARM). Due macchine con lo stesso `requirements.txt` installato possono continuare a comportarsi diversamente se differiscono in uno di questi strati -- esattamente il problema che la lezione 2 aveva lasciato annunciato e che risolve la sezione 4.

---
## 3. Pacchettizzare l'API della lezione 7 in un file reale

Scriviamo `api.py`: la stessa `Cliente`, lo stesso `/predecir` e `/predecir_lote` della lezione 7, ma come un modulo indipendente che carica `modelo.pkl` all'avvio -- esattamente ciò che un contenitore Docker eseguirebbe con `uvicorn`.

In [ ]:
API_PY_CODE = '\n"""API di churn (lezione 7), pacchettizzata per essere servita con uvicorn dentro un contenitore."""\nimport joblib\nimport pandas as pd\nfrom fastapi import FastAPI\nfrom pydantic import BaseModel, Field\nfrom typing import Literal, List\n\nCOLUMNAS_ENTRADA = ["plan", "contrato", "metodo_pago", "antiguedad_meses",\n                     "uso_mensual_horas", "tickets_soporte", "factura_mensual"]\n\nMODELO = joblib.load("modelo.pkl")\n\n\nclass Cliente(BaseModel):\n    id_cliente: str\n    antiguedad_meses: int = Field(ge=0)\n    plan: Literal["basico", "estandar", "premium"]\n    contrato: Literal["mensual", "anual"]\n    uso_mensual_horas: float = Field(ge=0)\n    tickets_soporte: int = Field(ge=0)\n    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]\n    factura_mensual: float = Field(ge=0)\n\n\ndef cliente_a_fila(cliente):\n    return pd.DataFrame([cliente.model_dump()])[COLUMNAS_ENTRADA]\n\n\napp = FastAPI(title="churn-api")\n\n\n@app.post("/predecir")\ndef predecir(cliente: Cliente):\n    fila = cliente_a_fila(cliente)\n    probabilidad = float(MODELO.predict_proba(fila)[0, 1])\n    return {\n        "id_cliente": cliente.id_cliente,\n        "probabilidad_abandono": round(probabilidad, 4),\n        "abandono_previsto": int(probabilidad >= 0.5),\n    }\n\n\n@app.post("/predecir_lote")\ndef predecir_lote(clientes: List[Cliente]):\n    filas = pd.DataFrame([c.model_dump() for c in clientes])[COLUMNAS_ENTRADA]\n    probabilidades = MODELO.predict_proba(filas)[:, 1]\n    return [\n        {"id_cliente": c.id_cliente, "probabilidad_abandono": round(float(p), 4), "abandono_previsto": int(p >= 0.5)}\n        for c, p in zip(clientes, probabilidades)\n    ]\n'

In [ ]:
RUTA_API = os.path.join(CARPETA, "api.py")
with open(RUTA_API, "w") as f:
    f.write(API_PY_CODE.strip() + "\n")
print("api.py scritto in:", RUTA_API, f"({len(API_PY_CODE.splitlines())} righe)")

---
## 4. Scrivere il `Dockerfile` reale

Un `Dockerfile` descrive, istruzione per istruzione, come costruire un'**immagine**: un filesystem completo (sistema operativo di base, Python, librerie di sistema, dipendenze Python e il nostro codice), pronto per essere eseguito allo stesso modo su qualsiasi macchina con Docker installato.

In [ ]:
DOCKERFILE_CODE = 'FROM python:3.12-slim\n\nWORKDIR /app\n\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY api.py .\nCOPY modelo.pkl .\n\nEXPOSE 8000\n\nCMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]\n'

In [ ]:
RUTA_DOCKERFILE = os.path.join(CARPETA, "Dockerfile")
with open(RUTA_DOCKERFILE, "w") as f:
    f.write(DOCKERFILE_CODE)
print("Dockerfile scritto in:", RUTA_DOCKERFILE)
print(DOCKERFILE_CODE)

| Istruzione | Cosa fa qui |
|---|---|
| `FROM python:3.12-slim` | fissa il sistema operativo di base E la versione di Python -- ciò che `requirements.txt` NON fissa mai |
| `WORKDIR /app` | crea ed entra in `/app` dentro l'immagine |
| `COPY requirements.txt .` + `RUN pip install ...` | installa le dipendenze DENTRO l'immagine, con le stesse librerie C di sistema Debian del `FROM` |
| `COPY api.py .` / `COPY modelo.pkl .` | inserisce il nostro codice e l'artefatto addestrato nell'immagine |
| `EXPOSE 8000` | documenta quale porta ascolterà il contenitore |
| `CMD [...]` | il comando eseguito all'avvio di un contenitore di questa immagine |

### 4.1 Validare la sintassi e la struttura del Dockerfile, in Python, senza Docker

In [ ]:
INSTRUCCIONES_VALIDAS = {
    "FROM", "WORKDIR", "COPY", "ADD", "RUN", "ENV", "EXPOSE", "CMD",
    "ENTRYPOINT", "ARG", "LABEL", "USER", "VOLUME", "STOPSIGNAL", "HEALTHCHECK", "SHELL",
}
# MAINTAINER è ufficialmente deprecato da Docker 1.13 (usare LABEL invece): lo trattiamo come proibito
INSTRUCCIONES_PROHIBIDAS = {"MAINTAINER"}


def validar_dockerfile(ruta):
    """Valida struttura e ordine del Dockerfile leggendo riga per riga, senza usare Docker."""
    instrucciones = []
    with open(ruta) as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith("#"):
                continue
            palabra = linea.split()[0].upper()
            instrucciones.append(palabra)

    errores = []
    if not instrucciones or instrucciones[0] != "FROM":
        errores.append("la prima istruzione deve essere FROM")
    prohibidas_usadas = [i for i in instrucciones if i in INSTRUCCIONES_PROHIBIDAS]
    if prohibidas_usadas:
        errores.append(f"istruzioni proibite trovate: {prohibidas_usadas}")
    desconocidas = [i for i in instrucciones if i not in INSTRUCCIONES_VALIDAS and i not in INSTRUCCIONES_PROHIBIDAS]
    if desconocidas:
        errores.append(f"istruzioni sconosciute: {desconocidas}")
    if "COPY" not in instrucciones and "ADD" not in instrucciones:
        errores.append("manca almeno un COPY/ADD per inserire codice nell'immagine")
    if "RUN" not in instrucciones:
        errores.append("manca almeno un RUN (qui, installare le dipendenze)")
    if not instrucciones or instrucciones[-1] not in ("CMD", "ENTRYPOINT"):
        errores.append("l'ultima istruzione deve essere CMD o ENTRYPOINT")
    idx_from = instrucciones.index("FROM") if "FROM" in instrucciones else -1
    idx_copy = next((i for i, x in enumerate(instrucciones) if x in ("COPY", "ADD")), len(instrucciones))
    idx_run = instrucciones.index("RUN") if "RUN" in instrucciones else len(instrucciones)
    if not (idx_from < idx_copy and idx_from < idx_run):
        errores.append("FROM deve precedere COPY/ADD e RUN")
    return instrucciones, errores


instrucciones_encontradas, errores_dockerfile = validar_dockerfile(RUTA_DOCKERFILE)
print("istruzioni, in ordine:", instrucciones_encontradas)
print("errori trovati:", errores_dockerfile)
assert errores_dockerfile == [], "il Dockerfile non ha superato la validazione di struttura"
print("Dockerfile valido: FROM per primo, COPY e RUN presenti, CMD alla fine, senza istruzioni proibite.")

**Cosa osservi?** La validazione non esegue nulla dentro un'immagine: legge il file riga per riga, ignora commenti e righe vuote, e verifica regole concrete (FROM per primo, almeno un COPY e un RUN, CMD/ENTRYPOINT alla fine, nessuna istruzione proibita come `MAINTAINER`). È un controllo di **struttura**, non che l'immagine si costruisca correttamente -- questo lo confermerebbe solo un `docker build` reale (sezione 9).

---
## 5. Il flusso di GitHub Actions: `ci.yml`

**CI (integrazione continua)** significa: ogni volta che qualcuno cambia il codice, un sistema automatico installa le dipendenze ed esegue la suite di test, senza intervento umano. GitHub Actions è il sistema di CI integrato in GitHub; un flusso (*workflow*) si descrive in un file YAML dentro `.github/workflows/`.

In [ ]:
CI_YML_CODE = 'name: CI - churn API\n\non:\n  push:\n    branches: [main]\n  pull_request:\n    branches: [main]\n\njobs:\n  test-and-register:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Scaricare il codice del repository\n        uses: actions/checkout@v4\n\n      - name: Configurare Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.12"\n\n      - name: Installare le dipendenze (API + test)\n        run: pip install -r requirements.txt pytest httpx\n\n      - name: Eseguire la suite di test (lezione 6)\n        run: pytest tests/ -q\n\n      - name: Registrare il modello (passo simbolico, lezione 4)\n        if: success()\n        run: echo "Test superati -- registrazione del modello come candidato nel registro di MLflow"\n'

In [ ]:
os.makedirs(os.path.join(CARPETA, ".github", "workflows"), exist_ok=True)
RUTA_CI_YML = os.path.join(CARPETA, ".github", "workflows", "ci.yml")
with open(RUTA_CI_YML, "w") as f:
    f.write(CI_YML_CODE)
print("ci.yml scritto in:", RUTA_CI_YML)
print(CI_YML_CODE)

| Sezione del YAML | Cosa significa |
|---|---|
| `name:` | il nome che vedrai nella tab "Actions" di GitHub |
| `on:` | gli eventi che innescano il flusso (qui: `push`/`pull_request` su `main`) |
| `jobs:` | uno o più lavori, ognuno con la propria macchina virtuale |
| `runs-on: ubuntu-latest` | quale sistema operativo usa la macchina virtuale del job |
| `steps:` | la lista ordinata di passi dentro il job |
| `uses: actions/checkout@v4` | un passo riutilizzato da un altro repository (qui, scaricare il codice) |
| `run: pytest tests/ -q` | un passo che esegue direttamente un comando di shell |
| `if: success()` | condizione: questo passo si esegue solo se TUTTI i passi precedenti hanno avuto successo |

### 5.1 Validare la sintassi e la struttura del ci.yml, in Python, con `yaml.safe_load`

In [ ]:
with open(RUTA_CI_YML) as f:
    flujo = yaml.safe_load(f.read())

# Attenzione: in YAML 1.1 (quello usato da PyYAML) la chiave "on" senza virgolette viene interpretata come il booleano True,
# non come la stringa "on" -- per questo controlliamo le due forme possibili della chiave
clave_on = "on" if "on" in flujo else (True if True in flujo else None)

errores_yml = []
if "name" not in flujo:
    errores_yml.append("manca la chiave 'name'")
if clave_on is None:
    errores_yml.append("manca la chiave 'on' (trigger)")
if "jobs" not in flujo or not flujo["jobs"]:
    errores_yml.append("manca la chiave 'jobs' o è vuota")
else:
    for nombre_job, job in flujo["jobs"].items():
        if "runs-on" not in job:
            errores_yml.append(f"il job '{nombre_job}' non ha 'runs-on'")
        if "steps" not in job or not job["steps"]:
            errores_yml.append(f"il job '{nombre_job}' non ha 'steps'")

print("chiavi di primo livello:", sorted(str(k) for k in flujo.keys()))
print("n. di step del job:", len(flujo["jobs"]["test-and-register"]["steps"]))
print("errori trovati:", errores_yml)
assert errores_yml == [], "il ci.yml non ha superato la validazione di struttura"
print("ci.yml valido: name, on, jobs, runs-on e steps presenti.")

**Cosa osservi?** `yaml.safe_load` fa un parsing REALE del file: se il YAML fosse malformato (indentazione sbagliata, due punti dove non dovrebbero esserci), questa cella lancerebbe un'eccezione prima di arrivare all'`assert`. Il controllo delle chiavi attese (`name`, `on`, `jobs`, `runs-on`, `steps`) confirma inoltre che il file ha la STRUTTURA che GitHub Actions si aspetta, non solo che è YAML valido in generale.

> **Attenzione (trappola reale di YAML):** in YAML 1.1 -- quello usato da PyYAML di default -- una chiave `on:` senza virgolette viene interpretata come il booleano `True`, non come la stringa `"on"`. Se cercassi `"on" in flujo` senz'altro, il controllo fallirebbe in silenzio anche se il file è scritto perfettamente bene. È un errore reale e documentato dalla comunità di GitHub Actions con strumenti basati su PyYAML, non una curiosità inventata per questo capitolo.

---
## 6. Scrivere i test che la CI eseguirebbe (lezione 6 + lezione 7)

Scriviamo `tests/test_modelo.py` (come nella lezione 6: l'F1 non deve scendere sotto una soglia) e `tests/test_api.py` (come nella lezione 7, con `TestClient`, su `api.py` per davvero).

In [ ]:
TEST_MODELO_PY = '\nimport joblib\nimport pandas as pd\nfrom sklearn.metrics import f1_score\n\nF1_MINIMO = 0.28\n\n\ndef test_modelo_supera_el_umbral_de_f1():\n    """Come nella lezione 6: il modello attuale non deve peggiorare sotto una soglia conosciuta."""\n    modelo = joblib.load("modelo.pkl")\n    df = pd.read_csv(RUTA_CSV_TESTS)\n    columnas = ["plan", "contrato", "metodo_pago", "antiguedad_meses",\n                "uso_mensual_horas", "tickets_soporte", "factura_mensual"]\n    predicciones = modelo.predict(df[columnas])\n    f1 = f1_score(df["abandono"], predicciones)\n    assert f1 >= F1_MINIMO\n'
TEST_API_PY = '\nfrom fastapi.testclient import TestClient\nfrom api import app\n\ncliente_api = TestClient(app)\n\nCLIENTE_VALIDO = {\n    "id_cliente": "C_TEST_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",\n    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,\n}\n\n\ndef test_predecir_con_cliente_valido_responde_200():\n    respuesta = cliente_api.post("/predecir", json=CLIENTE_VALIDO)\n    assert respuesta.status_code == 200\n    assert set(respuesta.json().keys()) == {"id_cliente", "probabilidad_abandono", "abandono_previsto"}\n\n\ndef test_predecir_con_plan_invalido_responde_422():\n    cliente_malo = dict(CLIENTE_VALIDO, plan="diamante")\n    respuesta = cliente_api.post("/predecir", json=cliente_malo)\n    assert respuesta.status_code == 422\n'

In [ ]:
# test_modelo.py ha bisogno del percorso assoluto al CSV perché pytest si esegue con proyecto_churn/ come directory corrente
ruta_csv_absoluta = os.path.abspath(RUTA_CSV)
texto_test_modelo = "RUTA_CSV_TESTS = " + repr(ruta_csv_absoluta) + "\n" + TEST_MODELO_PY

with open(os.path.join(CARPETA, "tests", "test_modelo.py"), "w") as f:
    f.write(texto_test_modelo)
with open(os.path.join(CARPETA, "tests", "test_api.py"), "w") as f:
    f.write(TEST_API_PY)
with open(os.path.join(CARPETA, "tests", "__init__.py"), "w") as f:
    f.write("")
print("test scritti in:", os.path.join(CARPETA, "tests"))

### 6.1 Eseguire la suite LOCALMENTE (l'unica cosa genuinamente eseguibile senza Docker/GitHub qui)

In [ ]:
# Questo è ESATTAMENTE ciò che la CI eseguirebbe nel passo "Ejecutar la suite de tests (clase 6)":
# pytest sulla cartella tests/, con conftest capace di importare api.py come modulo (sys.path include CARPETA)
sys.path.insert(0, os.path.abspath(CARPETA))
resultado_pytest = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    capture_output=True, text=True, cwd=CARPETA,
)
salida_pytest = resultado_pytest.stdout + resultado_pytest.stderr
m_passed = re.search(r"(\d+) passed", salida_pytest)
m_failed = re.search(r"(\d+) failed", salida_pytest)
resumen_tests = {
    "returncode": resultado_pytest.returncode,
    "passed": int(m_passed.group(1)) if m_passed else 0,
    "failed": int(m_failed.group(1)) if m_failed else 0,
}
print("riepilogo della suite (senza tempi, non sono deterministici):", resumen_tests)
assert resumen_tests["failed"] == 0 and resumen_tests["returncode"] == 0
print("I 3 test passano: 1 del modello (lezione 6) e 2 dell'API (lezione 7).")

**Cosa osservi?** Questo `subprocess.run(["pytest", ...])` è letteralmente lo stesso comando che il passo `run: pytest tests/ -q` di `ci.yml` eseguirebbe su una macchina virtuale di GitHub. La differenza è l'ambiente: qui viene eseguito nello stesso processo Python di questo notebook, non in un contenitore pulito -- per questo la sezione 9 dichiara esplicitamente che questo NON sostituisce l'esecuzione del flusso completo su un repository reale.

In [ ]:
# Lo stesso "if: success()" di ci.yml, ora in Python: continuiamo solo se la suite è passata
if resumen_tests["failed"] == 0:
    print("Test superati -- registrando il modello come candidato (sezione seguente).")
else:
    raise SystemExit("test falliti: NON si registra il modello")

---
## 7. Il registro dei modelli di MLflow: candidato, produzione, archiviato

Come nella lezione 4, l'archivio di MLflow è una cartella LOCALE dentro `proyecto_churn/`. Il **registro dei modelli** aggiunge sopra questo un concetto: un modello con NOME (`churn_modelo`), con diverse VERSIONI numerate, ognuna che punta a un run concreto. Un **alias** (`candidato`, `produccion`, `archivado`) punta a un'unica versione e si può riassegnare -- è il meccanismo attuale di MLflow (sostituisce i vecchi *stage*, ormai deprecati).

In [ ]:
RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)
NOMBRE_EXPERIMENTO = "churn_registro"
mlflow.set_experiment(NOMBRE_EXPERIMENTO)
NOMBRE_MODELO_REGISTRADO = "churn_modelo"

print("Tracking URI:", mlflow.get_tracking_uri())
assert mlflow.get_tracking_uri().startswith("file:") and "proyecto_churn" in mlflow.get_tracking_uri()

### 7.1 Registrare la versione 1 (già "in produzione") e la versione 2 (candidata)

In [ ]:
# Versione 1: la configurazione SENZA bilanciare (quella "già in produzione" prima di questa lezione, lezione 4)
clasificador_v1 = LogisticRegression(random_state=SEMILLA, max_iter=1000)
pipeline_v1 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v1)])
pipeline_v1.fit(X_train, y_train)
f1_v1 = round(f1_score(y_test, pipeline_v1.predict(X_test)), 4)
print("F1 della versione 1 (senza bilanciare):", f1_v1)

with mlflow.start_run(run_name="version_1_sin_balancear") as run_v1:
    mlflow.log_param("class_weight", "ninguno")
    mlflow.log_metric("f1", f1_v1)
    mlflow.sklearn.log_model(pipeline_v1, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_run_v1 = run_v1.info.run_id

version_v1 = mlflow.register_model(f"runs:/{id_run_v1}/modelo", NOMBRE_MODELO_REGISTRADO)
print("versione registrata:", version_v1.version)

In [ ]:
# Versione 2: la configurazione vincitrice (bilanciata) che questo notebook ha riaddestrato all'inizio
with mlflow.start_run(run_name="version_2_balanceada") as run_v2:
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("f1", F1)
    mlflow.sklearn.log_model(pipeline, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_run_v2 = run_v2.info.run_id

version_v2 = mlflow.register_model(f"runs:/{id_run_v2}/modelo", NOMBRE_MODELO_REGISTRADO)
print("versione registrata:", version_v2.version)

In [ ]:
cliente_mlflow = MlflowClient()

# Un "alias" punta a UNA versione concreta e si può riassegnare: è il meccanismo che sostituisce i vecchi "stage"
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "produccion", version_v1.version)
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "candidato", version_v2.version)
cliente_mlflow.set_model_version_tag(NOMBRE_MODELO_REGISTRADO, version_v2.version, "f1_test", str(F1))

version_en_produccion = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "produccion").version
version_candidata = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "candidato").version
print("alias 'produccion' -> versione:", version_en_produccion)
print("alias 'candidato' -> versione:", version_candidata)
assert version_en_produccion == version_v1.version and version_candidata == version_v2.version

**Cosa osservi?** Due versioni dello STESSO modello registrato (`churn_modelo`), ognuna con il proprio alias: la 1 in `produccion` (quella che ipoteticamente stava già servendo traffico), la 2 in `candidato` (appena addestrata, in attesa di validazione). Nessun cliente reale riceve ancora previsioni dalla versione 2 -- è esattamente ciò che separa CI da CD (sezione 9).

### 7.2 Promuovere il candidato a produzione (solo se la CI è passata E non peggiora la metrica)

In [ ]:
# Il passo "cauto" (sezione 9 del capitolo): il candidato si promuove SOLO perché i test della CI sono già passati
# (resumen_tests["failed"] == 0, verificato più sopra) E perché il suo F1 non è peggiore di quello della versione in produzione
f1_produccion_actual = float(cliente_mlflow.get_model_version(NOMBRE_MODELO_REGISTRADO, version_en_produccion).tags.get("f1_test", f1_v1))
puede_promover = resumen_tests["failed"] == 0 and F1 >= f1_produccion_actual - 1e-9
print("si può promuovere il candidato a produzione?", puede_promover, f"(F1 candidato={F1}, F1 produzione attuale={f1_produccion_actual})")
assert puede_promover

# Riassegnare l'alias "produccion" alla v2 lo TOGLIE automaticamente dalla v1 (un alias punta a una sola versione)
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "produccion", version_v2.version)
cliente_mlflow.delete_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "candidato")
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "archivado", version_v1.version)
cliente_mlflow.set_model_version_tag(NOMBRE_MODELO_REGISTRADO, version_v1.version, "estado", "archivado")

version_en_produccion_final = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "produccion").version
version_archivada = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "archivado").version
print("alias 'produccion' ora punta alla versione:", version_en_produccion_final)
print("alias 'archivado' punta alla versione:", version_archivada)
assert version_en_produccion_final == version_v2.version and version_archivada == version_v1.version
print("Ciclo di vita completo: v1 candidato->produzione->archiviato, v2 candidato->produzione.")

**Cosa osservi?** La promozione non è automatica "perché i test sono passati": verifica DUE condizioni (`resumen_tests["failed"] == 0` E `F1 >= f1_produccion_actual`) prima di spostare l'alias `produccion`. Riassegnare un alias lo TOGLIE automaticamente dalla versione precedente -- per questo, dopo aver promosso la v2, verifichiamo che `produccion` non punti più alla v1, e le assegniamo l'alias `archivado` al suo posto.

---
## 8. CI contro CD: perché in ML conviene un CD più cauto

**CI (integrazione continua)**: ogni cambio di codice innesca automaticamente installazione + test. Se qualcosa non va, si sa in pochi minuti, non alla prossima volta che qualcuno guarda il progetto. È esattamente ciò che fa `ci.yml` (sezioni 5 e 6).

**CD (distribuzione continua)**: se i test passano, il cambio viene pubblicato automaticamente -- nel software tradizionale, spesso senza nessun passo manuale tra "test verdi" e "in produzione".

In ML, promuovere un modello con i test in verde NON è sufficiente, per ragioni che non esistono nel software tradizionale:

| Rischio specifico di ML | Perché un test unitario non lo rileva |
|---|---|
| **Drift**: la distribuzione dei dati reali cambia col tempo (lezione 11) | i test della sezione 6 verificano il CSV di oggi, non i dati di tra tre mesi |
| Una metrica globale accettabile può nascondere un sottogruppo servito male | `f1_score` su tutto il set di test non dice nulla su un segmento specifico di clienti |
| Il "corretto" cambia col business, non solo col codice | un modello può restare tecnicamente corretto e non essere più la decisione di business adeguata |

Per questo, un CD cauto in ML aggiunge di solito, TRA "test verdi" e "servire il 100% del traffico": una promozione esplicita dell'alias (come la sezione 7.2, non automatica del tutto), un deployment **canary** o **shadow** (lezione 9) che confronta il candidato con la produzione attuale prima di sostituirla del tutto, e una revisione umana della metrica prima di spostare l'alias `produccion`. La sezione 7.2 di questo notebook è esattamente quel passo intermedio: automatizzato, ma con una condizione esplicita, non un `if: success()` nudo.

---
## 9. Provare il flusso completo fuori da questo notebook

> **Attenzione:** niente di quanto segue viene eseguito in questo notebook. Sono i comandi ESATTI che lanceresti sulla tua macchina o repository.
>
> **Con Docker installato in locale**, dentro `proyecto_churn/` (o la cartella dove hai `Dockerfile`, `api.py`, `modelo.pkl` e `requirements.txt`):
> ```
> docker build -t churn-api:1.0 .
> docker run -p 8000:8000 churn-api:1.0
> ```
> e in un altro terminale:
> ```
> curl -X POST http://localhost:8000/predecir -H "Content-Type: application/json" \
>   -d "{\"id_cliente\":\"C1\",\"antiguedad_meses\":12,\"plan\":\"estandar\",\"contrato\":\"mensual\",\"uso_mensual_horas\":15.0,\"tickets_soporte\":1,\"metodo_pago\":\"tarjeta\",\"factura_mensual\":22.5}"
> ```
>
> **Con un repository proprio su GitHub**: copia `.github/workflows/ci.yml` (e `requirements.txt`, `api.py`, `modelo.pkl`, `tests/`) in un repository, fai `git push`, e apri la tab **Actions** del repository su github.com. Vedrai il job `test-and-register` eseguirsi passo passo, dal vivo, con il risultato reale di `pytest` -- esattamente gli stessi passi che abbiamo validato nelle sezioni 5 e 6, ma in esecuzione vera sull'infrastruttura di GitHub.

### 9.1 C'è Docker installato in questo ambiente?

In [ ]:
resultado_which = subprocess.run(["which", "docker"], capture_output=True, text=True)
docker_disponible = resultado_which.returncode == 0
print("c'è un eseguibile 'docker' in questo ambiente?:", docker_disponible)
if docker_disponible:
    print("(percorso trovato, ma NON lo usiamo in questo notebook -- vedi il riquadro di Attenzione del capitolo)")
else:
    print("(non trovato in questo ambiente -- è quanto previsto in Colab/questo ambiente di validazione, vedi il riquadro di Attenzione)")

---
## 10. Pulizia finale della cartella di lavoro

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo

- Un `Dockerfile` reale (`FROM`/`WORKDIR`/`COPY`/`RUN`/`EXPOSE`/`CMD`) pacchettizza l'API della lezione 7 con il suo sistema operativo di base, Python e librerie C -- ciò che `requirements.txt` (lezione 2) non fissa mai.
- Un `ci.yml` reale di GitHub Actions installa le dipendenze, esegue `pytest` e, se passa, "registra" il modello (passo simbolico, `if: success()`).
- Entrambi i file sono stati VALIDATI per davvero in Python (struttura riga per riga del Dockerfile; `yaml.safe_load` + chiavi attese del `.yml`), senza aver bisogno di Docker né di GitHub installati.
- I test che la CI eseguirebbe sono stati eseguiti LOCALMENTE con `subprocess.run(["pytest", ...])`: è l'unica parte genuinamente eseguibile in questo ambiente.
- Il **registro dei modelli di MLflow** (alias `candidato`/`produccion`/`archivado`, riassegnabili) ha segnato il ciclo di vita completo di due versioni dello stesso modello.
- **CI** testa automaticamente ogni cambio; **CD** lo pubblica automaticamente se passa. In ML conviene un CD più cauto: drift, sottogruppi serviti male e decisioni di business sono rischi che un test in verde non rileva da solo.

*Continua con la lezione 9: Distribuzione e pubblicazione.*